In [36]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage

import logging
import os
import time

In [5]:
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)


In [18]:
def tools_used(messages):
    """Return all tools called by the agent on the current turn."""
    last_human = max(
        i for i, m in enumerate(messages)
        if isinstance(m, HumanMessage)
    )

    tools = []

    for m in messages[last_human + 1:]:
        for call in getattr(m, "tool_calls", []) or []:
            tools.append(call["name"])

    return tools


In [38]:

NOTES = [
    "LangGraph gives an agent memory using a checkpointer, and a thread_id names one conversation. "
    "Same thread_id remembers earlier turns; a new thread_id starts fresh.",

    "A tool is an ordinary Python function with a docstring. The model reads the name, docstring "
    "and typed arguments to decide when and how to call it.",

    "create_agent(model, tools) builds the whole ReAct loop for you: it calls the model, runs the "
    "tool it asks for, feeds the result back, and repeats until done.",

    "RAG (retrieval-augmented generation) means: retrieve relevant text first, then let the model "
    "answer using that text, so answers are grounded in your documents instead of guessed.",

    "Saarathi Academy runs a 12-week AI Engineering and Machine Learning course, two hours a day, "
    "in Old Baneshwor, Kathmandu.",

    "The safe way to run arithmetic from a model is a small ast-based evaluator, never Python's "
    "eval(), because a tool is a door into your system.",
]


In [49]:


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key="AQ.Ab8RN6IYAdxQ7F89RG3xrdztFbyEU4ZqBhRvKZA_8UG5PLmE1A",  
)



# from langchain_openrouter import ChatOpenRouter

# model = ChatOpenRouter(
#     model="stealth/ox-alpha",
#     api_key="sk-or-..."
# )

In [50]:
_vec = TfidfVectorizer().fit(NOTES)          # build the index once
_M   = _vec.transform(NOTES)


@tool
def search_notes(query: str) -> str:
    """Search the user's course notes and return the most relevant note."""
    return NOTES[int(cosine_similarity(_vec.transform([query]), _M)[0].argmax())]

@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression, e.g. '8 * 12'."""
    return safe_eval(expression)

@tool
def save_note(text: str) -> str:
    """Save a fact the user wants remembered for later."""
    NOTEBOOK.append(text); return "saved"



In [41]:
TOOLS = [search_notes, calculator, save_note]

In [40]:
SYSTEM_PROMPT = """

You are a helpful research assistant. Answer concisely and cite sources when possible. My name is Alex

"""

In [51]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(model, tools=TOOLS,
                         checkpointer=InMemorySaver(), system_prompt=SYSTEM_PROMPT)   # tools + memory

cfg = {"configurable": {"thread_id": "aug_31st_test_class_v1"}}   # one conversation

# raw_response = assistant.invoke({"messages": [{"role": "user",
#                   "content": "what is my name"}]}, me)

# print("done")

In [1]:



"""

1. rag tool
2. internet search
3. basic maths
4. file changing (create, modify, delete)
5. code execution

"""

'\n\n1. rag tool\n2. internet search\n3. basic maths\n4. file changing (create, modify, delete)\n5. code execution\n\n'

In [46]:
TESTSET = [  
    {"q": "What is my name?",            "tool": None,           "expect": "alex"},
    {"q": "What is 12 * 3?",             "tool": "calculator",   "expect": "36"},
    {"q": "How does agent memory work?", "tool": "search_notes", "expect": "checkpointer"},
    {"q": "Just say hi.",                "tool": None,           "expect": "hi"},
]


In [52]:
def get_text(content):
    """Extract text from a LangChain message content."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        texts = []

        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                texts.append(item.get("text", ""))

        return " ".join(texts)

    return str(content)


def is_human_message(message):
    """Check whether a message is a human/user message."""
    return message.__class__.__name__ == "HumanMessage"


def tools_used(messages):
    """Return all tools called on the current turn."""

    # Find the last user message
    human_indices = [
        i for i, m in enumerate(messages)
        if is_human_message(m)
    ]

    if not human_indices:
        return []

    last_human = human_indices[-1]

    # Only inspect messages after the latest user message
    tools = []

    for message in messages[last_human + 1:]:
        tool_calls = getattr(message, "tool_calls", None) or []

        for tool_call in tool_calls:
            if isinstance(tool_call, dict):
                name = tool_call.get("name")

                if name:
                    tools.append(name)

    return tools


answer_ok = 0
tool_ok = 0

for case in TESTSET:

    print(f"\nTesting: {case['q']}")

    out = assistant.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": case["q"]
                }
            ]
        },
        cfg
    )

    messages = out["messages"]

    # -------------------------
    # Final answer
    # -------------------------
    answer = get_text(messages[-1].content).lower()

    # -------------------------
    # Tools used
    # -------------------------
    used_tools = tools_used(messages)

    # -------------------------
    # Evaluate answer
    # -------------------------
    answer_pass = case["expect"].lower() in answer

    # -------------------------
    # Evaluate tool
    # -------------------------
    expected_tool = case["tool"]

    if expected_tool is None:
        tool_pass = len(used_tools) == 0
    else:
        tool_pass = expected_tool in used_tools

    answer_ok += answer_pass
    tool_ok += tool_pass

    # -------------------------
    # Debug output
    # -------------------------
    print("Answer:", answer)
    print("Expected:", case["expect"])
    print("Expected tool:", expected_tool)
    print("Used tools:", used_tools)
    print("Answer OK:", answer_pass)
    print("Tool OK:", tool_pass)

    print("Sleeping for 2 seconds...")
    time.sleep(2)


# -------------------------
# Final scores
# -------------------------

print("\n" + "=" * 40)

print(
    f"Answer Score: "
    f"{answer_ok}/{len(TESTSET)} "
    f"({answer_ok / len(TESTSET):.2%})"
)

print(
    f"Tool Score: "
    f"{tool_ok}/{len(TESTSET)} "
    f"({tool_ok / len(TESTSET):.2%})"
)



Testing: What is my name?
Answer: your name is alex.
Expected: alex
Expected tool: None
Used tools: []
Answer OK: True
Tool OK: True
Sleeping for 2 seconds...

Testing: What is 12 * 3?
Answer: 12 * 3 is 36.
Expected: 36
Expected tool: calculator
Used tools: ['calculator']
Answer OK: True
Tool OK: True
Sleeping for 2 seconds...

Testing: How does agent memory work?
Answer: based on your course notes, agent memory (specifically in langgraph) works using a **checkpointer**:

* **`thread_id`:** this identifier represents a single conversation session.
* **same `thread_id`:** the agent remembers previous turns within that conversation.
* **new `thread_id`:** starts a fresh conversation with no memory of earlier interactions.
Expected: checkpointer
Expected tool: search_notes
Used tools: ['search_notes']
Answer OK: True
Tool OK: True
Sleeping for 2 seconds...

Testing: Just say hi.
Answer: hi, alex!
Expected: hi
Expected tool: None
Used tools: []
Answer OK: True
Tool OK: True
Sleeping for 2

In [25]:
out

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='e74e7688-f1f1-4350-adb5-2340dd41eeaf'),
  AIMessage(content=[{'type': 'text', 'text': 'Your name is Alex.', 'extras': {'signature': 'EskCCsYCARFNMg/Jk4JsmlMxUW2ya5N7+5n7Kd0o1ZTkPkAbNvw+/dQ8/3N/t3mWsZ9oAqf/1208xFfEPVE5MX6VUjA9XaU4Gd47DysXVZKP4/RoZVL/nNLJl41j/BoO73HkDdHu74VM7INLNFB3saueeDTKexT7jqaXuqQQ1RLkFSWPHfmT+oe0/Ti3YM7hQAQJDV2hG6Bhgivxo6O0eQsgqkCj9lVc7I9p54+pvDRjkZEZ35j16FACvxyDgF10uXnZGypBofkn/UqBsTi/kio4Ffu0qpsq0fBDji0ivjAupg4fK3ShXrrH8nNinURRaOI0Y4lmkgzDR/L8AKnaSlU8xlBSHoN7xupI7gnZqwfRWSPYOJqrqmrv8GlSeLTb/7svCqWcYh4bloCkMVUdSHUqHOCoB5uau+A0zXQMy/F+lkTiNKROmw6dnDs='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05790-7fcd-71a2-93fc-141bfbf7d8ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 57, 'total_t

In [26]:
test_out = dict(out.copy())

In [27]:
test_out

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='e74e7688-f1f1-4350-adb5-2340dd41eeaf'),
  AIMessage(content=[{'type': 'text', 'text': 'Your name is Alex.', 'extras': {'signature': 'EskCCsYCARFNMg/Jk4JsmlMxUW2ya5N7+5n7Kd0o1ZTkPkAbNvw+/dQ8/3N/t3mWsZ9oAqf/1208xFfEPVE5MX6VUjA9XaU4Gd47DysXVZKP4/RoZVL/nNLJl41j/BoO73HkDdHu74VM7INLNFB3saueeDTKexT7jqaXuqQQ1RLkFSWPHfmT+oe0/Ti3YM7hQAQJDV2hG6Bhgivxo6O0eQsgqkCj9lVc7I9p54+pvDRjkZEZ35j16FACvxyDgF10uXnZGypBofkn/UqBsTi/kio4Ffu0qpsq0fBDji0ivjAupg4fK3ShXrrH8nNinURRaOI0Y4lmkgzDR/L8AKnaSlU8xlBSHoN7xupI7gnZqwfRWSPYOJqrqmrv8GlSeLTb/7svCqWcYh4bloCkMVUdSHUqHOCoB5uau+A0zXQMy/F+lkTiNKROmw6dnDs='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05790-7fcd-71a2-93fc-141bfbf7d8ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 57, 'total_t

In [33]:
a = 5
b = a
b= b+2

print(a, b)

5 7


5